# ✂️ บทที่ 2 — เตรียม "แบบฝึกหัด" ให้ AI (Tiling & Augmentation)

> **สื่อการสอน Workshop:** การจำแนกพื้นที่ปลูกปาล์มน้ำมันด้วยภาพถ่ายดาวเทียมและ AI  
> ต่อจาก: [บทที่ 1 — รู้จักข้อมูล](01_read_and_explore_v2.ipynb)

---

## บทที่ 2 นี้ เราจะทำอะไร?

ในบทที่ 1 เราเห็นแล้วว่าภาพ SAR ทั้งภาพมีขนาดใหญ่มาก (ระดับจังหวัด)  
คอมพิวเตอร์ไม่สามารถ "มอง" ภาพใหญ่ขนาดนั้นพร้อมกันได้ทั้งหมดในครั้งเดียว  
เราจึงต้อง **ตัดภาพเป็นชิ้นเล็กๆ** (Tiles) เพื่อให้ AI ค่อยๆ เรียนรู้ทีละชิ้น

<!-- 🖼️ โครงสร้างสำหรับสร้างรูปประกอบ (ผู้สอนจะนำไปสร้างภาพจริงมาใส่แทน) -->
**โครงสร้างรูป:**
```
ภาพ SAR ต้นฉบับ (ใหญ่มาก)
  ┌──────────────┐
  │              │
  │              │      ✂️ ตัดเป็นชิ้นเล็กๆ (256x256)
  │              │    ─────────────────────────→    [ ][ ][ ]
  │              │                                  [ ][ ][ ]
  └──────────────┘                                  [ ][ ][ ]
  (5004 x 5004 px)                                 (หลายหมื่น tiles)
```

**Prompt:**
`"A premium minimalist infographic illustrating the concept of image tiling. On the left, a large satellite map icon labeled 'ภาพ SAR ต้นฉบับ'. In the middle, a sleek scissors icon cutting across. On the right, a neat grid of small square image tiles (256x256) labeled 'Tiles (แบบฝึกหัด AI)'. Light theme, crisp white background #FFFFFF, mint-green accents #35C778, flat vector illustration, 16:9, no device frames."`

ในบทนี้เราจะ:
1. ทำความเข้าใจ **Tiling, Stride และ Overlap**
2. เรียนรู้การเพิ่มข้อมูลด้วย **Augmentation** (พลิก หมุน ภาพ)
3. ตัด Tiles สำหรับ 1 เดือน (เพื่อทดสอบ)
4. ตัด Tiles รวมหลายเดือน (Combined Dataset) สำหรับใช้สอน AI ในบทที่ 3

---

## ทฤษฎี: Stride และ Overlap (การตัดแบบซ้อนทับ)

ถ้าเราตัดภาพต่อกันพอดี (เหมือนหั่นเต๋า) แปลงปาล์มที่อยู่ตรง "รอยต่อ" จะถูกตัดขาด  
AI จะเรียนรู้รูปร่างของแปลงได้ยาก

**วิธีแก้:** เราต้องเลื่อนกรอบตัดให้ **ซ้อนทับกัน (Overlap)**
- **Tile Size:** ขนาดของกรอบตัด (ในที่นี้คือ 256 x 256 pixel)
- **Stride:** ระยะที่กรอบเลื่อนไปในแต่ละครั้ง (ในที่นี้คือ 128 pixel)

![2.1.png](https://raw.githubusercontent.com/A-Phrom-A/ssb-workshop/main/https://raw.githubusercontent.com/A-Phrom-A/ssb-workshop/main/img/2.1.png)

In [ ]:
# =====================================================================
# ⚠️ สำหรับรันบน Local Machine (เครื่องตัวเอง)
# =====================================================================
import os
BASE_DIR = r'C:\Users\kittisak\Downloads\ssb tutor\drive_structure\ch02_prepare_tiles'


In [ ]:
# =====================================================================
# ☁️ สำหรับรันบน Google Colab (คอมเมนต์ไว้ก่อน)
# =====================================================================
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_DIR = '/content/drive/MyDrive/ssb_workshop/ch02_prepare_tiles'


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1.1 — ติดตั้งเครื่องมือ (เพิ่ม geoai และ albumentations)
# ═══════════════════════════════════════════════════════════
# geoai: เครื่องมือสำหรับตัดภาพและเตรียมข้อมูล
# albumentations: เครื่องมือสำหรับทำ Augmentation

!pip install geoai-py albumentations rasterio geopandas scikit-learn --quiet

print("✅ ติดตั้ง geoai และเครื่องมืออื่นๆ เสร็จสิ้น")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1.2 — Import และกำหนดตำแหน่งไฟล์
# ═══════════════════════════════════════════════════════════

import os
import glob
import re
import shutil
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import geoai
import albumentations as A


In [ ]:
# ── Paths เหมือนบทที่ 1 ──
RASTER_DIR  = os.path.join(BASE_DIR, "data")
VECTOR_PATH = os.path.join(BASE_DIR, "data", "ground_truth.geojson")

# ── โฟลเดอร์สำหรับเก็บผลลัพธ์ (Tiles) ──
OUTPUT_DIR  = os.path.join(BASE_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✅ กำหนดตำแหน่งไฟล์เรียบร้อย ผลลัพธ์จะเก็บที่: {OUTPUT_DIR}")

---

## ขั้นตอนที่ 2 — แยกข้อมูล Train และ Hold-out

เราต้องแบ่งภาพ SAR เป็น 2 กลุ่ม:
1. **Train Set:** ภาพที่เราจะตัดเป็น Tiles เพื่อสอน AI ( เดือน M01, M02, M03, M04)
2. **Hold-out Set:** ภาพที่เรา **ไม่ให้ AI เห็นเลย** ระหว่างการเรียนรู้ เอาไว้เป็น "ข้อสอบปลายภาค" ( เดือน M05)

> ⚠️ **ห้ามให้ข้อมูลปนกัน (Data Leakage):** ถ้าข้อเฉลยสอบรั่วไปอยู่ในแบบฝึกหัด AI จะท่องจำข้อสอบและได้คะแนนดีเกินจริง

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2.1 — แยกไฟล์ตามชื่อเดือน
# ═══════════════════════════════════════════════════════════

# เดือนที่เป็นข้อสอบปลายภาค (Hold-out)
HOLDOUT_YEAR_MONTHS = {"2026_M05"}

train_raster_files = []
holdout_raster_files = []

sar_files = sorted(glob.glob(os.path.join(RASTER_DIR, "Sentinel-1*.tif")))

for f in sar_files:
    basename = os.path.basename(f)
    # ดึงปีและเดือนจากชื่อไฟล์ เช่น 2026_M01
    m = re.search(r"(\d{4})_(M\d{2})", basename)
    if m:
        year_month = f"{m.group(1)}_{m.group(2)}"
        if year_month in HOLDOUT_YEAR_MONTHS:
            holdout_raster_files.append(f)
        else:
            train_raster_files.append(f)

print(f"📊 พบภาพ SAR ทั้งหมด {len(sar_files)} ไฟล์")
print(f"   → เป็น Train Set (แบบฝึกหัด): {len(train_raster_files)} ไฟล์")
for f in train_raster_files:
    print(f"      • {os.path.basename(f)}")
    
print(f"   → เป็น Hold-out Set (ข้อสอบ): {len(holdout_raster_files)} ไฟล์")
for f in holdout_raster_files:
    print(f"      • {os.path.basename(f)}")

---

## ขั้นตอนที่ 3 — ทฤษฎี Data Augmentation

**Data Augmentation** คือการเพิ่มจำนวน "แบบฝึกหัด" โดยการนำ Tile เดิมมา พลิก, หมุน, เบลอ  
ช่วยให้ AI ได้เห็นต้นปาล์มในหลายๆ มุมมอง และเก่งขึ้น (ไม่ท่องจำ)

![2.2.png](https://raw.githubusercontent.com/A-Phrom-A/ssb-workshop/main/https://raw.githubusercontent.com/A-Phrom-A/ssb-workshop/main/img/2.2.png)

> 💡 **ใน Workshop นี้:** เราจะสร้าง Augmentation 10 แบบต่อ 1 Tile

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3.1 — กำหนดรูปแบบการทำ Augmentation
# ═══════════════════════════════════════════════════════════
# ใช้ Albumentations library — กำหนดว่าให้มันทำอะไรกับภาพได้บ้าง

transform = A.Compose([
    A.HorizontalFlip(p=0.5),           # พลิกซ้าย-ขวา (โอกาส 50%)
    A.VerticalFlip(p=0.5),             # พลิกบน-ล่าง (โอกาส 50%)
    A.RandomRotate90(p=0.5),           # หมุน 90 องศา (โอกาส 50%)
    A.GaussianBlur(blur_limit=(3,5), p=0.1), # เบลอเล็กน้อย (โอกาส 10%)
])


> 💡 **เกร็ดความรู้: พารามิเตอร์ `p=0.5` คืออะไร?**

เมื่อเรากำหนด `p=0.5` หมายความว่า ทุกๆ ครั้งที่มีรูปภาพถูกส่งเข้ามาในกระบวนการ (Pipeline) ระบบจะทำการ **"ทอยเหรียญ"** (สุ่มความน่าจะเป็น) เพื่อตัดสินใจว่าจะทำคำสั่งนั้นกับภาพหรือไม่ 

**ตัวอย่างการทำงาน:**
- **🖼️ รูปที่ 1** เดินทางมาถึงคำสั่ง `A.HorizontalFlip(p=0.5)`

     **ระบบสุ่มได้ค่า 0.7 (เกิน 0.5) ❌ ไม่พลิกรูป**
- **🖼️ รูปที่ 2** เดินทางมาถึงคำสั่งเดิม

     **ระบบสุ่มได้ค่า 0.2 (น้อยกว่า 0.5) ✅ พลิกรูป**

**✨ ประโยชน์:**
การตั้งค่าแบบนี้ทำให้ข้อมูล (Dataset) ของเรามีความหลากหลาย (Augmentation) มากขึ้น โมเดลจะไม่ได้เห็นภาพแบบเดิมซ้ำๆ ในแต่ละรอบการฝึกสอน (Epoch) ซึ่งช่วยลดปัญหาโมเดลท่องจำข้อมูล (Overfitting) ได้เป็นอย่างดี


In [ ]:
# ── กำหนดค่าคงที่สำหรับการตัด Tiles ──
TILE_SIZE = 256      # ขนาด Tile = 256x256 pixel
STRIDE = 128         # เลื่อนกรอบทีละ 128 pixel (Overlap 50%)
AUG_COUNT = 10       # ทำ Augmentation 10 ภาพ ต่อ 1 Tile เดิม

print("✅ เตรียมเงื่อนไขการทำ Augmentation และ Tiling พร้อมแล้ว")
print(f"   ขนาดภาพ: {TILE_SIZE}x{TILE_SIZE}, ระยะเลื่อน: {STRIDE}, สร้างเพิ่ม: {AUG_COUNT} เท่า")

---

## ขั้นตอนที่ 4 — สร้าง Combined Dataset (หลายเดือน)

เราจะลูป (loop) นำภาพ Train Set ทีละเดือน มาตัดเป็น Tiles ด้วย `geoai.export_geotiff_tiles`  
แล้วนำมารวมกันในโฟลเดอร์เดียว (Combined) เพื่อนำไปสอน AI

> ⏱️ **ขั้นตอนนี้ใช้เวลาประมาณ 2-5 นาที ต่อ 1 ภาพ**

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4.1 — ตัดและรวม Tiles ทุกภาพใน Train Set
# ═══════════════════════════════════════════════════════════

COMBINED_IMAGES = os.path.join(OUTPUT_DIR, "images")
COMBINED_LABELS = os.path.join(OUTPUT_DIR, "labels")
os.makedirs(COMBINED_IMAGES, exist_ok=True)
os.makedirs(COMBINED_LABELS, exist_ok=True)

print(f"✂️ เริ่มกระบวนการ Tiling ({len(train_raster_files)} ภาพ)")
print("=" * 60)

for raster_path in train_raster_files:
    basename = os.path.basename(raster_path)
    prefix = os.path.splitext(basename)[0]  # ชื่อภาพเพื่อใช้เป็นรหัสนำหน้า Tile
    
    print(f"\nกำลังตัดภาพ: {basename}")
    
    # ใช้ geoai ตัดภาพ SAR และ GeoJSON พร้อมๆ กัน
    geoai.export_geotiff_tiles(
        in_raster=raster_path,
        in_class_data=VECTOR_PATH,          # geoai จะแปลง CRS ให้อัตโนมัติ
        out_folder=OUTPUT_DIR,
        
        tile_size=TILE_SIZE,              # 256
        stride=STRIDE,                    # 128
        augmentation_count=AUG_COUNT,     # 10
        augmentation_transforms=transform,
                      # เติมชื่อภาพข้างหน้าชื่อ Tile จะได้ไม่ซ้ำกัน
        quiet=True,
    )
    print(f"✅ ตัดภาพ {basename} เสร็จแล้ว")

print("\n" + "=" * 60)
print("✅ การสร้าง Combined Dataset เสร็จสมบูรณ์!")

---

## ขั้นตอนที่ 5 — ตรวจสอบผลลัพธ์ (Tiles)

เพื่อความมั่นใจ เราจะสุ่มดูภาพที่ถูกตัดออกมาว่าหน้าตาเป็นอย่างไร

> **สิ่งที่เราควรจะเห็น:**
> - `images`: ภาพขาวดำ 2 bands (VV, VH) ขนาด 256x256
> - `labels`: ภาพขาวดำ 1 band (0=ไม่ใช่ปาล์ม, 1=ปาล์ม)
> - ถ้าเอามาซ้อนกัน ส่วนที่เป็นปาล์ม (1) ควรจะทับอยู่บนพื้นที่ขาวสว่าง (ปาล์มสะท้อนคลื่นดี) ในภาพ SAR พอดี

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5.1 — สรุปจำนวน Tiles ทั้งหมด
# ═══════════════════════════════════════════════════════════

n_images = len(glob.glob(os.path.join(COMBINED_IMAGES, "*.tif")))
n_labels = len(glob.glob(os.path.join(COMBINED_LABELS, "*.tif")))

print("📊 สรุป Combined Dataset:")
print(f"   ภาพ SAR (images): {n_images:,} tiles")
print(f"   ป้ายคำตอบ (labels): {n_labels:,} tiles")
if n_images == n_labels and n_images > 0:
    print("   ✅ จำนวนตรงกัน สมบูรณ์แบบ!")
else:
    print("   ❌ จำนวนไม่ตรงกัน หรือยังไม่ได้ตัดภาพ ให้ตรวจสอบขั้นตอนที่ 4 อีกครั้ง")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5.2 — สุ่มดูตัวอย่าง Tiles (Image + Label + Overlay)
# ═══════════════════════════════════════════════════════════

import random

def stretch_percentile(band, p_low=2, p_high=98):
    vmin, vmax = np.nanpercentile(band, [p_low, p_high])
    return np.clip((band - vmin) / (vmax - vmin + 1e-6), 0, 1)

# สุ่มไฟล์ภาพ 3 ไฟล์
all_image_paths = sorted(glob.glob(os.path.join(COMBINED_IMAGES, "*.tif")))
if len(all_image_paths) > 0:
    sample_paths = random.sample(all_image_paths, min(3, len(all_image_paths)))
    
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    fig.suptitle("สุ่มดูตัวอย่าง Tiles (SAR vs Ground Truth)", fontsize=16, fontweight="bold")
    
    for i, img_path in enumerate(sample_paths):
        basename = os.path.basename(img_path)
        lbl_path = os.path.join(COMBINED_LABELS, basename)
        
        # อ่าน SAR
        with rasterio.open(img_path) as src_img:
            sar = src_img.read()  # (2, 256, 256)
        
        # อ่าน Label
        if os.path.exists(lbl_path):
            with rasterio.open(lbl_path) as src_lbl:
                lbl = src_lbl.read(1)  # (256, 256)
        else:
            lbl = np.zeros((TILE_SIZE, TILE_SIZE))
            
        # ภาพ SAR VV
        vv_stretched = stretch_percentile(sar[0])
        axes[i, 0].imshow(vv_stretched, cmap="gray")
        axes[i, 0].set_title(f"Tile {i+1}\nSAR (VV)")
        axes[i, 0].axis("off")
        
        # ภาพ SAR VH
        vh_stretched = stretch_percentile(sar[1])
        axes[i, 1].imshow(vh_stretched, cmap="gray")
        axes[i, 1].set_title("SAR (VH)")
        axes[i, 1].axis("off")
        
        # ภาพ Label (0/1)
        axes[i, 2].imshow(lbl, cmap="Greens")
        axes[i, 2].set_title("Ground Truth (เขียว=ปาล์ม)")
        axes[i, 2].axis("off")
        
        # ภาพ Overlay
        overlay = np.stack([vv_stretched, vv_stretched, vv_stretched], axis=-1)
        # ระบายสีเขียวทับบริเวณที่เป็นปาล์ม (lbl == 1)
        overlay[lbl == 1] = [0.2, 0.8, 0.3]
        
        axes[i, 3].imshow(overlay)
        axes[i, 3].set_title("Overlay")
        axes[i, 3].axis("off")
        
    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.show()
else:
    print("⚠️ ไม่พบภาพในโฟลเดอร์ ให้รันขั้นตอนที่ 4 ก่อน")

---

## สรุปบทที่ 2

เราได้เตรียมข้อมูลเสร็จสมบูรณ์ พร้อมสำหรับการนำไปสอน AI ในบทต่อไปแล้ว!

---

### 📚 ไปต่อ → [บทที่ 3 — สอน AI ให้รู้จักปาล์ม (Train Model)](03_train_model.ipynb)